# Customer Spending Prediciton

In [1]:
#import necessary libraries
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
#data collection
df = pd.read_csv("ShopSmart_data.csv")
df

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent
0,46,620358.0,11.0,8,Silver,Beauty,Tier 1,Yes,16924.36
1,27,560886.0,13.0,7,Silver,Fashion,Tier 1,No,16467.52
2,52,866655.0,11.0,9,Bronze,Home,Tier 1,No,19224.61
3,55,816016.0,17.0,9,Silver,Home,Tier 2,Yes,20219.63
4,18,355468.0,14.0,0,Bronze,Home,Tier 3,Yes,7345.37
...,...,...,...,...,...,...,...,...,...
1995,47,557297.0,5.0,6,Gold,Home,Tier 1,No,17423.64
1996,46,1027242.0,11.0,12,Bronze,Fashion,Tier 3,No,19939.64
1997,40,677690.0,15.0,4,Bronze,Fashion,Tier 3,Yes,13397.79
1998,34,565410.0,11.0,8,Silver,Electronics,Tier 3,Yes,17682.11


### EDA

In [3]:
#check top few rows
df.head()

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent
0,46,620358.0,11.0,8,Silver,Beauty,Tier 1,Yes,16924.36
1,27,560886.0,13.0,7,Silver,Fashion,Tier 1,No,16467.52
2,52,866655.0,11.0,9,Bronze,Home,Tier 1,No,19224.61
3,55,816016.0,17.0,9,Silver,Home,Tier 2,Yes,20219.63
4,18,355468.0,14.0,0,Bronze,Home,Tier 3,Yes,7345.37


In [4]:
#check bottom few rows
df.tail()

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent
1995,47,557297.0,5.0,6,Gold,Home,Tier 1,No,17423.64
1996,46,1027242.0,11.0,12,Bronze,Fashion,Tier 3,No,19939.64
1997,40,677690.0,15.0,4,Bronze,Fashion,Tier 3,Yes,13397.79
1998,34,565410.0,11.0,8,Silver,Electronics,Tier 3,Yes,17682.11
1999,65,397581.0,6.0,8,Bronze,Fashion,Tier 3,No,11165.42


In [5]:
#check info of dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 2000 non-null   int64  
 1   Annual_Income       1984 non-null   float64
 2   Website_Visits      1984 non-null   float64
 3   Previous_Purchases  2000 non-null   int64  
 4   Membership_Type     1984 non-null   str    
 5   Preferred_Category  1984 non-null   str    
 6   City_Tier           2000 non-null   str    
 7   Discount_Used       2000 non-null   str    
 8   Amount_Spent        2000 non-null   float64
dtypes: float64(3), int64(2), str(4)
memory usage: 182.7 KB


In [6]:
#check missing values
df.isnull().sum()

Age                    0
Annual_Income         16
Website_Visits        16
Previous_Purchases     0
Membership_Type       16
Preferred_Category    16
City_Tier              0
Discount_Used          0
Amount_Spent           0
dtype: int64

In [7]:
#Handle missing values
avg_income = round(df["Annual_Income"].mean())
avg_web_visits= round(df["Website_Visits"].mean())
freq_membership = df["Membership_Type"].mode()[0]
freq_category  = df["Preferred_Category"].mode()[0]

df["Annual_Income"] = df["Annual_Income"].fillna(avg_income)
df["Website_Visits"] = df["Website_Visits"].fillna(avg_web_visits)
df["Membership_Type"] = df["Membership_Type"].fillna(freq_membership)
df["Preferred_Category"] = df["Preferred_Category"].fillna(freq_category)

In [8]:
df.isnull().sum()

Age                   0
Annual_Income         0
Website_Visits        0
Previous_Purchases    0
Membership_Type       0
Preferred_Category    0
City_Tier             0
Discount_Used         0
Amount_Spent          0
dtype: int64

In [9]:
#check duplicate rows
df[df.duplicated()]

,Age,Annual_Income,Website_Visits,Previous_Purchases,Membership_Type,Preferred_Category,City_Tier,Discount_Used,Amount_Spent


In [10]:
#check summary of dataset
df.describe()

,Age,Annual_Income,Website_Visits,Previous_Purchases,Amount_Spent
count,2000.000000,2.000000e+03,2000.00000,2000.0000,2000.000000
mean,40.917500,7.525418e+05,9.95650,6.9655,17505.982405
std,12.780116,2.728578e+05,3.09134,2.6235,3866.061235
min,18.000000,1.800000e+05,2.00000,0.0000,6875.410000
25%,32.000000,5.605328e+05,8.00000,5.0000,14864.417500
50%,41.000000,7.520560e+05,10.00000,7.0000,17381.955000
75%,50.000000,9.346820e+05,12.00000,9.0000,19927.317500
max,65.000000,1.604606e+06,20.00000,16.0000,35371.200000


In [11]:
#check numeric data
num_cols = []
for col in df.columns:
    if df[col].dtype!='str':
        num_cols.append(col)
df[num_cols]

,Age,Annual_Income,Website_Visits,Previous_Purchases,Amount_Spent
0,46,620358.0,11.0,8,16924.36
1,27,560886.0,13.0,7,16467.52
2,52,866655.0,11.0,9,19224.61
3,55,816016.0,17.0,9,20219.63
4,18,355468.0,14.0,0,7345.37
...,...,...,...,...,...
1995,47,557297.0,5.0,6,17423.64
1996,46,1027242.0,11.0,12,19939.64
1997,40,677690.0,15.0,4,13397.79
1998,34,565410.0,11.0,8,17682.11


Scaling should be done on these numeric columns

In [12]:
#check object  data
for col in df.columns:
    if df[col].dtype=='str':
        print(col,"      :",list(df[col].unique()))

Membership_Type       : ['Silver', 'Bronze', 'Gold']
Preferred_Category       : ['Beauty', 'Fashion', 'Home', 'Electronics']
City_Tier       : ['Tier 1', 'Tier 2', 'Tier 3']
Discount_Used       : ['Yes', 'No']


Encoding needs to be done on these object columns

### Model Pipeline

In [13]:
#Divide the data into input and output features
X = df.drop("Amount_Spent",axis=1)
y = df.Amount_Spent

In [14]:
#Split training and testing dataset
Xtrain,Xtest,ytrain,ytest = train_test_split(X,y,test_size=0.3,random_state=42)

In [15]:
#Separate numeric and object columns
num_cols = Xtrain.select_dtypes(include="number").columns
cat_cols = Xtrain.select_dtypes(include="str").columns

In [16]:
#multi model pipelines
models = {"Linear Regression": Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),cat_cols),
                                                                            ("scaler",StandardScaler(),num_cols)])),
                                         ("model",LinearRegression())]),
          "Ridge": Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),cat_cols),
                                                                            ("scaler",StandardScaler(),num_cols)])),
                                         ("model",Ridge(alpha=50))]),
          "Lasso": Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),cat_cols),
                                                                            ("scaler",StandardScaler(),num_cols)])),
                                         ("model",Lasso(alpha=100))]),
          "KNN": Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),cat_cols),
                                                                            ("scaler",StandardScaler(),num_cols)])),
                                         ("model",KNeighborsRegressor(n_neighbors=6))]),
          "Decision Tree": Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),cat_cols),
                                                                            ("scaler",StandardScaler(),num_cols)])),
                                         ("model",DecisionTreeRegressor(max_depth=10))]),
          "Random Forest": Pipeline([("preprocessor",ColumnTransformer([("encoder",OneHotEncoder(),cat_cols),
                                                                            ("scaler",StandardScaler(),num_cols)])),
                                         ("model",RandomForestRegressor(n_estimators=100,max_depth=6,random_state=42))]),

}

In [17]:
#multi - model training
model_perf = {}
for model_name,pipeline in models.items():
    pipeline.fit(Xtrain,ytrain)
    ypred = pipeline.predict(Xtest)
    r2 = r2_score(ytest,ypred)
    mse = mean_squared_error(ytest,ypred)
    mae = mean_absolute_error(ytest,ypred)
    model_perf[model_name] = {"R2":r2,"MSE":mse,"MAE":mae}

In [18]:
#model performance table
perf_df = pd.DataFrame(model_perf)
perf_df

,Linear Regression,Ridge,Lasso,KNN,Decision Tree,Random Forest
R2,0.948411,0.945508,9.219361e-01,8.305840e-01,7.717924e-01,8.286898e-01
MSE,748773.147905,790913.976980,1.133044e+06,2.458957e+06,3.312277e+06,2.486451e+06
MAE,617.571794,637.820955,8.214268e+02,1.221249e+03,1.401779e+03,1.241418e+03


In [19]:
#fetch best model
best_model = max(model_perf.items(),key = lambda t:t[1]["R2"] )
print(best_model)
best_model_name = best_model[0]
best_pipeline = models[best_model_name]

('Linear Regression', {'R2': 0.9484114030748212, 'MSE': 748773.1479047127, 'MAE': 617.5717936465328})


In [20]:
#predictions
best_pipeline.predict(Xtest)

array([14300.70821746, 18868.39803924, 21971.15353009, 18737.94405993,
       16436.42119351, 17202.76154403, 15377.20150439, 21325.65246883,
       15865.06206069, 17277.42577381, 13888.14976577, 16093.40582274,
       13677.31782594, 17415.88257148, 15663.74260122, 10125.80259628,
       18838.29969687, 22163.78213348, 15654.46952326, 15196.37727142,
       19751.23780891, 18291.28036664, 17704.59403663, 17981.41677218,
       15263.22950359, 20738.0649449 , 17682.11238067, 17959.57952301,
       18354.74055789, 12602.2609369 , 15750.38863006, 13783.45246416,
       28358.7905974 , 17406.7892587 , 16593.83373872, 15833.40324013,
       16347.84898046, 18796.29947389, 14010.96415369, 20601.09709545,
       16641.1926763 ,  9845.65563136, 16165.36509138, 23541.32722822,
       19370.07059054, 20790.03422243, 16708.21877095, 17124.57748792,
       19334.39449236, 23177.65065194, 18636.64185262, 12833.74443307,
        7984.53431687, 24789.60762942, 20556.17864924, 13783.75242183,
      

In [21]:
#save best model pipeline
import pickle
with open("best_model_pipeline.pkl","wb") as file:
    pickle.dump(best_pipeline,file)

In [22]:
#save cleaned df
df.to_csv("cleaned_data.csv")

In [23]:
Xtrain.columns

Index(['Age', 'Annual_Income', 'Website_Visits', 'Previous_Purchases',
       'Membership_Type', 'Preferred_Category', 'City_Tier', 'Discount_Used'],
      dtype='str')